In [0]:
from pyspark.sql.functions import col,current_timestamp,input_file_name
import re

In [0]:
display(dbutils.fs.ls("/Volumes/investment_pyspark/bronze/landing/dividend/dividends_2025-08-30_to_2026-08-29.csv"))

In [0]:
source_path = "/Volumes/investment_pyspark/bronze/landing/dividend/"
schema_path = "/Volumes/investment_pyspark/bronze/landing/dividend/dividends_schema/"
checkpoint_path = "/Volumes/investment_pyspark/bronze/landing/dividend/dividends_checkpoint/"

# Clear stale schema and checkpoint for a fresh start
dbutils.fs.rm(schema_path, True)
dbutils.fs.rm(checkpoint_path, True)

# Drop existing table to avoid duplicates on reprocessing
#spark.sql("DROP TABLE IF EXISTS investment_pyspark.bronze.dividends_raw")

def clean_column_names(col_name):
    col_name = col_name.strip()
    col_name = re.sub(r"[^a-zA-Z0-9_]","_",col_name)
    col_name = re.sub(r"_+", "_",col_name)
    return col_name.strip('_')

df_bronze = (spark.readStream.format("cloudFiles")
.option("cloudFiles.format", "csv")
.option("cloudFiles.schemaLocation", schema_path)
.option("pathGlobFilter", "*.csv")
.option("header","true")
.option("inferSchema","true")
.load(source_path)
)
df_cleaned = df_bronze.toDF(*[clean_column_names(c) for c in df_bronze.columns])
df_transformed = df_cleaned.withColumn(
    "_ingestion_timestamp",current_timestamp()
).withColumn("_source_file",col("_metadata.file_path"))

query= (
    df_transformed.writeStream.format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("investment_pyspark.bronze.dividends_raw")
)
query.awaitTermination()

In [0]:
%sql
select * from investment_pyspark.bronze.dividends_raw;